# Alpaca Hourly LSTM Machine Learning Pipeline

This notebook pulls hourly historical stock data for `["MU", "GOOG", "TSLA", "SPY"]` from the Alpaca API, constructs a binary cross-sectional outperformance target, builds 240-hour sequence features, and trains and evaluates a baseline LSTM model against standard baselines.

In [1]:
import os
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame, TimeFrameUnit
from alpaca.data.enums import DataFeed
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping

RANDOM_STATE = 42
tf.keras.utils.set_random_seed(RANDOM_STATE)

# Load Alpaca Credentials safely from .env or fallback
api_key = None
secret_key = None
for env_path in ['../.env', '.env']:
    if os.path.exists(env_path):
        with open(env_path, 'r') as f:
            for line in f:
                if line.startswith("ALPACA_API_KEY="):
                    api_key = line.split("=")[1].strip()
                elif line.startswith("ALPACA_SECRET_KEY="):
                    secret_key = line.split("=")[1].strip()
                    
if not api_key:
    print("Warning: Using fallback hardcoded credentials.")
    api_key = "PKEN63LDIFPD43FZN5JQXXXHXV"
    secret_key = "3pmEQxuC9R5vGL4y6YSfDMWq9fgSM8nBZae8QGHyoE3C"
else:
    print("Successfully loaded Alpaca API keys from .env file!")

Successfully loaded Alpaca API keys from .env file!


In [2]:
class AlpacaDataPuller:
    def __init__(self, symbols, api_key, secret_key):
        self.symbols = symbols
        self.client = StockHistoricalDataClient(api_key, secret_key)
        
    def pull_hourly_data(self, start_date):
        data_list = []
        for symbol in self.symbols:
            print(f"Downloading hourly data for {symbol} starting from {start_date}...")
            request = StockBarsRequest(
                symbol_or_symbols=symbol,
                timeframe=TimeFrame(1, TimeFrameUnit.Hour),
                start=start_date,
                end=datetime.now(),
                feed=DataFeed.IEX
            )
            bars = self.client.get_stock_bars(request)
            data_list.append(bars.df)
            
        df_all = pd.concat(data_list).reset_index()
        df_all = df_all.rename(columns={
            'timestamp': 'timestamp',
            'symbol': 'symbol',
            'open': 'open',
            'high': 'high',
            'low': 'low',
            'close': 'close',
            'volume': 'volume'
        })
        return df_all

In [3]:
symbols = ["MU", "GOOG", "TSLA", "SPY"]
puller = AlpacaDataPuller(symbols, api_key, secret_key)

# Pull data starting 5 years back to ensure we have a massive hourly dataset
start_date = datetime.now() - timedelta(days=5 * 365)
df_raw = puller.pull_hourly_data(start_date)

# Sort chronologically per symbol
df_raw = df_raw.sort_values(by=['symbol', 'timestamp']).reset_index(drop=True)

# Calculate hourly percentage returns and shift to get Next_Return
df_raw['Return'] = df_raw.groupby('symbol')['close'].pct_change()
df_raw['Next_Return'] = df_raw.groupby('symbol')['Return'].shift(-1)
df_raw = df_raw.dropna(subset=['Return', 'Next_Return']).copy()

print(f"\nRetrieved {len(df_raw)} total hourly data points.")
print(df_raw.head())


Retrieved 37926 total hourly data points.
  symbol                 timestamp     open  ...         vwap    Return  Next_Return
1   GOOG 2021-07-23 14:00:00+00:00  2704.82  ...  2723.284165  0.011108     0.003695
2   GOOG 2021-07-23 15:00:00+00:00  2733.83  ...  2735.044231  0.003695     0.003650
3   GOOG 2021-07-23 16:00:00+00:00  2742.28  ...  2743.202122  0.003650     0.004622
4   GOOG 2021-07-23 17:00:00+00:00  2751.06  ...  2758.496751  0.004622    -0.000152
5   GOOG 2021-07-23 18:00:00+00:00  2765.58  ...  2770.180182 -0.000152    -0.003057

[5 rows x 11 columns]


In [4]:
# 1. Define Binary relative target (Outperform = 1, Underperform = 0 relative to hourly median return)
median_returns = df_raw.groupby('timestamp')['Next_Return'].transform('median')
df_raw['Target'] = (df_raw['Next_Return'] >= median_returns).astype(int)

# 2. Chronological Splitting (70/15/15) to prevent target leakage
unique_dates = sorted(df_raw['timestamp'].unique())
N = len(unique_dates)

TRAIN_FRAC = 0.70
VAL_FRAC = 0.15

TRAIN_END = unique_dates[int(N * TRAIN_FRAC)]
VAL_END = unique_dates[int(N * (TRAIN_FRAC + VAL_FRAC))]

train_df = df_raw[df_raw['timestamp'] <= TRAIN_END].copy()
val_df = df_raw[(df_raw['timestamp'] > TRAIN_END) & (df_raw['timestamp'] <= VAL_END)].copy()
test_df = df_raw[df_raw['timestamp'] > VAL_END].copy()

# 3. Standardize Return ONLY using training-set statistics per symbol
train_stats = train_df.groupby('symbol')['Return'].agg(['mean', 'std']).reset_index()
train_stats = train_stats.rename(columns={'mean': 'mu_train', 'std': 'sigma_train'})

# Merge stats onto all partitions
train_df = train_df.merge(train_stats, on='symbol', how='left')
val_df = val_df.merge(train_stats, on='symbol', how='left')
test_df = test_df.merge(train_stats, on='symbol', how='left')

# Calculate standardized returns
train_df['Std_Return'] = (train_df['Return'] - train_df['mu_train']) / train_df['sigma_train']
val_df['Std_Return'] = (val_df['Return'] - val_df['mu_train']) / val_df['sigma_train']
test_df['Std_Return'] = (test_df['Return'] - test_df['mu_train']) / test_df['sigma_train']

print(f"Train samples: {len(train_df)} | Val samples: {len(val_df)} | Test samples: {len(test_df)}")

Train samples: 26421 | Val samples: 5631 | Test samples: 5874


In [5]:
def create_lstm_sequences(df_split, window_size=240):
    sequences = []
    targets = []
    
    # We sort by symbol and timestamp to ensure alignment
    df_split = df_split.sort_values(by=['symbol', 'timestamp']).reset_index(drop=True)
    
    for symbol, group in df_split.groupby('symbol'):
        values = group['Std_Return'].values
        labels = group['Target'].values
        
        for i in range(window_size, len(values)):
            sequences.append(values[i-window_size:i])
            targets.append(labels[i])
            
    X = np.expand_dims(np.array(sequences), axis=-1)
    y = np.array(targets)
    return X, y

# Generate sequences for all splits
X_train, y_train = create_lstm_sequences(train_df)
X_val, y_val = create_lstm_sequences(val_df)
X_test, y_test = create_lstm_sequences(test_df)

print(f"X_train shape: {X_train.shape} | y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape} | y_test shape: {y_test.shape}")

X_train shape: (25461, 240, 1) | y_train shape: (25461,)
X_test shape: (4914, 240, 1) | y_test shape: (4914,)


In [6]:
# 1. Flatten inputs for baseline models
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_test_flat = X_test.reshape(X_test.shape[0], -1)

# 2. Dummy Classifier
dummy = DummyClassifier(strategy="most_frequent").fit(X_train_flat, y_train)
dummy_pred = dummy.predict(X_test_flat)

# 3. Logistic Regression
lr = LogisticRegression(max_iter=1000, C=1.0, random_state=RANDOM_STATE)
lr.fit(X_train_flat, y_train)
lr_pred = lr.predict(X_test_flat)

# 4. Random Forest
rf = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=RANDOM_STATE)
rf.fit(X_train_flat, y_train)
rf_pred = rf.predict(X_test_flat)

# 5. LSTM Classifier
lstm_model = Sequential([
    Input(shape=(X_train.shape[1], X_train.shape[2])), # (240, 1)
    LSTM(25, dropout=0.1, recurrent_dropout=0.1),
    Dense(2, activation="softmax")
])

lstm_model.compile(
    optimizer="rmsprop",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

early_stopping = EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)

print("Training LSTM on Hourly Alpaca Dataset...")
history = lstm_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=128,
    callbacks=[early_stopping],
    verbose=1
)

lstm_probs = lstm_model.predict(X_test)
lstm_pred = np.argmax(lstm_probs, axis=1)

Training LSTM on Hourly Alpaca Dataset...
Epoch 1/30
199/199 ━━━━━━━━━━━━━━━━━━━━ 7s 32ms/step - accuracy: 0.5155 - loss: 0.6928 - val_accuracy: 0.5177 - val_loss: 0.6926
Epoch 2/30
199/199 ━━━━━━━━━━━━━━━━━━━━ 6s 31ms/step - accuracy: 0.5151 - loss: 0.6926 - val_accuracy: 0.5174 - val_loss: 0.6925
Epoch 3/30
199/199 ━━━━━━━━━━━━━━━━━━━━ 6s 32ms/step - accuracy: 0.5158 - loss: 0.6927 - val_accuracy: 0.5189 - val_loss: 0.6924
Epoch 4/30
199/199 ━━━━━━━━━━━━━━━━━━━━ 6s 32ms/step - accuracy: 0.5160 - loss: 0.6927 - val_accuracy: 0.5185 - val_loss: 0.6925
Epoch 5/30
199/199 ━━━━━━━━━━━━━━━━━━━━ 6s 32ms/step - accuracy: 0.5159 - loss: 0.6926 - val_accuracy: 0.5174 - val_loss: 0.6924
Epoch 6/30
199/199 ━━━━━━━━━━━━━━━━━━━━ 6s 32ms/step - accuracy: 0.5166 - loss: 0.6926 - val_accuracy: 0.5194 - val_loss: 0.6924
Epoch 7/30
199/199 ━━━━━━━━━━━━━━━━━━━━ 6s 32ms/step - accuracy: 0.5177 - loss: 0.6925 - val_accuracy: 0.5202 - val_loss: 0.6924
Epoch 8/30
199/199 ━━━━━━━━━━━━━━━━━━━━ 7s 33ms/step - 

In [7]:
print("\n=============================================")
print("UNCONDITIONAL CLASSIFICATION PERFORMANCE")
print("=============================================")
models = {
    "Majority Class Baseline": dummy_pred,
    "Logistic Regression": lr_pred,
    "Random Forest": rf_pred,
    "LSTM Classifier": lstm_pred
}
for name, preds in models.items():
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds, average="macro", zero_division=0)
    print(f"{name:<25} : Accuracy = {acc:.4f} | Macro F1 = {f1:.4f}")

# 1. Align predictions and symbols using cumcount mask
test_results = test_df.sort_values(by=['symbol', 'timestamp']).copy()
mask = test_results.groupby('symbol').cumcount() >= 240
test_results = test_results[mask].copy()
test_results = test_results.reset_index(drop=True)

# Get probability scores for all models
eval_df = test_results.copy()
eval_df['LR_Prob'] = lr.predict_proba(X_test_flat)[:, 1]
eval_df['RF_Prob'] = rf.predict_proba(X_test_flat)[:, 1]
eval_df['LSTM_Prob'] = lstm_probs[:, 1]

def compute_ranking_accuracy(df, prob_col):
    correct_long = 0
    correct_short = 0
    total_days = 0
    
    for timestamp, group in df.groupby('timestamp'):
        if len(group) < 2:
            continue
        total_days += 1
        
        sorted_group = group.sort_values(by=prob_col, ascending=False)
        
        # Top 1 Long
        top_1 = sorted_group.iloc[0]
        if top_1['Target'] == 1:
            correct_long += 1
            
        # Bottom 1 Short
        bottom_1 = sorted_group.iloc[-1]
        if bottom_1['Target'] == 0:
            correct_short += 1
            
    return correct_long / total_days, correct_short / total_days, (correct_long + correct_short) / (2 * total_days)

print("\n=============================================")
print("DAILY RANKED PORTFOLIO ACCURACY (k=1)")
print("=============================================")
for model_name, prob_col in [("Logistic Regression", "LR_Prob"), ("Random Forest", "RF_Prob"), ("LSTM Classifier", "LSTM_Prob")]:
    long_acc, short_acc, overall_acc = compute_ranking_accuracy(eval_df, prob_col)
    print(f"{model_name:<25} : Long Acc = {long_acc:.2%} | Short Acc = {short_acc:.2%} | Overall Acc = {overall_acc:.2%}")



UNCONDITIONAL CLASSIFICATION PERFORMANCE
Majority Class Baseline   : Accuracy = 0.5151 | Macro F1 = 0.3400
Logistic Regression       : Accuracy = 0.5002 | Macro F1 = 0.4784
Random Forest             : Accuracy = 0.5140 | Macro F1 = 0.3661
LSTM Classifier           : Accuracy = 0.5108 | Macro F1 = 0.3717

DAILY RANKED PORTFOLIO ACCURACY (k=1)
Logistic Regression       : Long Acc = 50.62% | Short Acc = 48.84% | Overall Acc = 49.73%
Random Forest             : Long Acc = 50.77% | Short Acc = 47.37% | Overall Acc = 49.07%
LSTM Classifier           : Long Acc = 51.32% | Short Acc = 45.74% | Overall Acc = 48.53%
